In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import sqlite3

BASE = '/content/drive/MyDrive/ecommerce-analytics'

# Load clean sales data
df = pd.read_csv(f'{BASE}/data/processed/stage2_sales_clean.csv',
                 parse_dates=['InvoiceDate'])

# Load customers data (for customer-level queries)
df_cust = pd.read_csv(f'{BASE}/data/processed/stage2_customers.csv',
                      parse_dates=['InvoiceDate'])

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load DataFrames as SQL tables
df.to_sql('sales', conn, index=False, if_exists='replace')
df_cust.to_sql('customers', conn, index=False, if_exists='replace')

print("SQL engine ready.")
print(f"Table 'sales'     : {len(df):,} rows")
print(f"Table 'customers' : {len(df_cust):,} rows")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_178/65341023.py:11: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{BASE}/data/processed/stage2_sales_clean.csv',


SQL engine ready.
Table 'sales'     : 1,007,913 rows
Table 'customers' : 797,885 rows


In [ ]:
def run_sql(query, rows=10):
    result = pd.read_sql_query(query, conn)
    print(f"Rows returned: {len(result)}")
    return result.head(rows)

In [ ]:
monthly_revenue = run_sql("""
    SELECT
        Year,
        Month,
        ROUND(SUM(TotalRevenue), 2)   AS revenue,
        COUNT(DISTINCT Invoice)        AS num_orders,
        COUNT(DISTINCT "Customer ID")  AS unique_customers,
        ROUND(SUM(TotalRevenue) /
              COUNT(DISTINCT Invoice), 2) AS avg_order_value
    FROM sales
    GROUP BY Year, Month
    ORDER BY Year, Month
""", rows=30)

print(monthly_revenue.to_string(index=False))

Rows returned: 25
 Year  Month    revenue  num_orders  unique_customers  avg_order_value
 2009     12  796498.88        1682               955           473.54
 2010      1  616158.37        1105               720           557.61
 2010      2  529360.95        1201               772           440.77
 2010      3  754691.76        1681              1057           448.95
 2010      4  654735.26        1462               942           447.84
 2010      5  637197.87        1500               966           424.80
 2010      6  692173.82        1645              1041           420.77
 2010      7  640252.70        1529               928           418.74
 2010      8  671536.21        1425               911           471.25
 2010      9  856587.34        1839              1145           465.79
 2010     10 1095474.22        2301              1497           476.09
 2010     11 1408826.82        2747              1607           512.86
 2010     12  775459.06        1559               885      

In [ ]:
product_revenue = run_sql("""
    SELECT
        StockCode,
        Description,
        ROUND(SUM(TotalRevenue), 2)    AS total_revenue,
        SUM(Quantity)                  AS total_units_sold,
        COUNT(DISTINCT Invoice)        AS num_orders,
        ROUND(AVG(Price), 2)           AS avg_price
    FROM sales
    GROUP BY StockCode, Description
    ORDER BY total_revenue DESC
""", rows=20)

# Calculate cumulative revenue % (Pareto analysis)
product_revenue_full = pd.read_sql_query("""
    SELECT StockCode, Description,
           ROUND(SUM(TotalRevenue),2) AS total_revenue
    FROM sales
    GROUP BY StockCode, Description
    ORDER BY total_revenue DESC
""", conn)

total_rev = product_revenue_full['total_revenue'].sum()
product_revenue_full['revenue_%'] = (
    product_revenue_full['total_revenue'] / total_rev * 100
).round(2)
product_revenue_full['cumulative_%'] = (
    product_revenue_full['revenue_%'].cumsum().round(2)
)

top_80 = product_revenue_full[
    product_revenue_full['cumulative_%'] <= 80
]
print(f"Products driving 80% of revenue: {len(top_80):,}")
print(f"Out of total products           : {len(product_revenue_full):,}")
print(f"\nTop 10 products:")
print(product_revenue_full.head(10).to_string(index=False))

Rows returned: 5630
Products driving 80% of revenue: 1,208
Out of total products           : 5,630

Top 10 products:
StockCode                        Description  total_revenue  revenue_%  cumulative_%
    22423           REGENCY CAKESTAND 3 TIER      330590.32       1.71          1.71
   85123A WHITE HANGING HEART T-LIGHT HOLDER      244297.66       1.26          2.97
      DOT                     DOTCOM POSTAGE      166820.16       0.86          3.83
    47566                      PARTY BUNTING      148318.28       0.77          4.60
   85099B            JUMBO BAG RED RETROSPOT      142521.83       0.74          5.34
    84879      ASSORTED COLOUR BIRD ORNAMENT      117913.89       0.61          5.95
     POST                            POSTAGE      114079.63       0.59          6.54
    22086    PAPER CHAIN KIT 50'S CHRISTMAS       110735.44       0.57          7.11
    79321                      CHILLI LIGHTS       80540.88       0.42          7.53
   85099F               JUMBO BAG

In [ ]:
country_analysis = run_sql("""
    SELECT
        Country,
        ROUND(SUM(TotalRevenue), 2)    AS total_revenue,
        COUNT(DISTINCT Invoice)        AS num_orders,
        COUNT(DISTINCT "Customer ID")  AS unique_customers,
        ROUND(AVG(TotalRevenue), 2)    AS avg_order_value,
        ROUND(SUM(TotalRevenue) * 100.0 /
            (SELECT SUM(TotalRevenue) FROM sales), 2) AS revenue_pct
    FROM sales
    WHERE Country != 'United Kingdom'
    GROUP BY Country
    ORDER BY total_revenue DESC
""", rows=15)

print(country_analysis.to_string(index=False))

Rows returned: 42
        Country  total_revenue  num_orders  unique_customers  avg_order_value  revenue_pct
           EIRE      637222.13         626                 5            37.15         3.29
    Netherlands      543826.79         228                22           106.95         2.81
        Germany      422094.75         789               107            25.69         2.18
         France      327303.51         622                95            24.00         1.69
      Australia      166621.67          95                15            93.14         0.86
          Spain      106413.71         154                41            29.06         0.55
    Switzerland       99995.62          93                22            32.03         0.52
         Sweden       88358.64         105                19            66.14         0.46
        Belgium       63999.24         149                29            20.95         0.33
       Portugal       53022.29          95                24            

In [ ]:
peak_times = run_sql("""
    SELECT
        Day,
        Hour,
        COUNT(DISTINCT Invoice)       AS num_orders,
        ROUND(SUM(TotalRevenue), 2)   AS revenue
    FROM sales
    GROUP BY Day, Hour
    ORDER BY num_orders DESC
""", rows=15)

print(peak_times.to_string(index=False))

Rows returned: 85
      Day  Hour  num_orders   revenue
Wednesday    12        1221 550101.36
 Thursday    12        1153 520391.34
  Tuesday    12        1120 481996.02
Wednesday    13        1047 429608.11
  Tuesday    13        1042 481470.07
   Monday    12        1003 426076.75
 Thursday    13        1000 401893.78
   Sunday    12         975 344828.69
Wednesday    14         960 401844.99
   Friday    12         949 424771.51
   Monday    13         942 447899.30
 Thursday    14         918 436103.04
  Tuesday    14         860 399396.96
 Thursday    15         859 456401.14
 Thursday    10         858 486347.74


In [ ]:
top_customers = run_sql("""
    SELECT
        "Customer ID",
        Country,
        ROUND(SUM(TotalRevenue), 2)    AS total_spent,
        COUNT(DISTINCT Invoice)        AS num_orders,
        ROUND(AVG(TotalRevenue), 2)    AS avg_order_value,
        MIN(DATE(InvoiceDate))         AS first_purchase,
        MAX(DATE(InvoiceDate))         AS last_purchase
    FROM customers
    GROUP BY "Customer ID", Country
    ORDER BY total_spent DESC
""", rows=15)

print(top_customers.to_string(index=False))

Rows returned: 5955
 Customer ID        Country  total_spent  num_orders  avg_order_value first_purchase last_purchase
     18102.0 United Kingdom    570380.61         153           543.22     2009-12-01    2011-12-09
     14646.0    Netherlands    523342.07         164           134.54     2009-12-02    2011-12-08
     14156.0           EIRE    296063.44         202            71.89     2009-12-01    2011-11-30
     14911.0           EIRE    265757.91         510            23.22     2009-12-01    2011-12-08
     17450.0 United Kingdom    231390.55          61           521.15     2010-09-27    2011-12-01
     13694.0 United Kingdom    190020.84         164           122.52     2009-12-04    2011-12-06
     17511.0 United Kingdom    168491.62          85            81.12     2009-12-02    2011-12-07
     12415.0      Australia    143269.29          33           144.72     2010-06-30    2011-11-15
     16684.0 United Kingdom    141502.25          65           188.92     2009-12-07    2

In [ ]:
aov_trend = run_sql("""
    SELECT
        Year,
        Month,
        ROUND(SUM(TotalRevenue) /
              COUNT(DISTINCT Invoice), 2)  AS avg_order_value,
        COUNT(DISTINCT Invoice)            AS num_orders
    FROM sales
    GROUP BY Year, Month
    ORDER BY Year, Month
""", rows=30)

print(aov_trend.to_string(index=False))

Rows returned: 25
 Year  Month  avg_order_value  num_orders
 2009     12           473.54        1682
 2010      1           557.61        1105
 2010      2           440.77        1201
 2010      3           448.95        1681
 2010      4           447.84        1462
 2010      5           424.80        1500
 2010      6           420.77        1645
 2010      7           418.74        1529
 2010      8           471.25        1425
 2010      9           465.79        1839
 2010     10           476.09        2301
 2010     11           512.86        2747
 2010     12           497.41        1559
 2011      1           537.91        1086
 2011      2           466.11        1100
 2011      3           481.67        1454
 2011      4           408.22        1246
 2011      5           443.03        1681
 2011      6           468.25        1533
 2011      7           476.35        1475
 2011      8           534.17        1361
 2011      9           555.40        1837
 2011     10    

In [ ]:
results = {
    'monthly_revenue': pd.read_sql_query("""
        SELECT Year, Month,
               ROUND(SUM(TotalRevenue),2) AS revenue,
               COUNT(DISTINCT Invoice) AS num_orders
        FROM sales GROUP BY Year, Month
        ORDER BY Year, Month""", conn),

    'top_products': pd.read_sql_query("""
        SELECT StockCode, Description,
               ROUND(SUM(TotalRevenue),2) AS total_revenue,
               SUM(Quantity) AS units_sold
        FROM sales GROUP BY StockCode, Description
        ORDER BY total_revenue DESC LIMIT 50""", conn),

    'country_revenue': pd.read_sql_query("""
        SELECT Country,
               ROUND(SUM(TotalRevenue),2) AS total_revenue,
               COUNT(DISTINCT Invoice) AS num_orders
        FROM sales GROUP BY Country
        ORDER BY total_revenue DESC""", conn),

    'top_customers': pd.read_sql_query("""
        SELECT "Customer ID",
               ROUND(SUM(TotalRevenue),2) AS total_spent,
               COUNT(DISTINCT Invoice) AS num_orders
        FROM customers GROUP BY "Customer ID"
        ORDER BY total_spent DESC LIMIT 100""", conn),
}

BASE = '/content/drive/MyDrive/ecommerce-analytics'

for name, result_df in results.items():
    path = f'{BASE}/data/processed/sql_{name}.csv'
    result_df.to_csv(path, index=False)
    print(f"Saved to Drive: sql_{name}.csv ({len(result_df)} rows)")

print("\nStage 3 DONE!")

Saved to Drive: sql_monthly_revenue.csv (25 rows)
Saved to Drive: sql_top_products.csv (50 rows)
Saved to Drive: sql_country_revenue.csv (43 rows)
Saved to Drive: sql_top_customers.csv (100 rows)

Stage 3 DONE!
